This notebook shows sample inference using exported PnC onnx models

In [ ]:
import onnxruntime as ort
from transformers import AutoTokenizer
import numpy as np
import time

options = ort.SessionOptions()
options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

session = ort.InferenceSession("/mnt/e/pnc/final/PNCMODEL_bert-base_en-US_4.0_quantized_weights.onnx", sess_options=options)
print("session ok")

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "catalan is a western romance language and is the indigenous and official language of three autonomous communities in eastern spain"
tokens = tokenizer.encode(text)

mask = [1]*len(tokens)
token_type_ids = [1]*len(tokens)

print(len(tokens), tokens)

tensor_tokens = np.expand_dims(tokens, 0)
tensor_mask = np.expand_dims(mask, 0)
tensor_type = np.expand_dims(token_type_ids, 0)

output_names = ["punct_logits", "capit_logits"]
inputs = {
    "input_ids": tensor_tokens,
    "token_type_ids": tensor_type,
    "attention_mask": tensor_mask
}

num_loop = 10
total_diff = 0
for i in range(num_loop):
    t0 = time.time_ns()
    output = session.run(output_names, inputs)
    t1 = time.time_ns()
    total_diff += (t1 - t0)

total_diff /= num_loop
print("Average inference time: {:.3f} sec".format(total_diff/1e9))

punct_logits = output[0][0]
capit_logits = output[1][0]

punct = np.argmax(punct_logits, axis=1)
capit = np.argmax(capit_logits, axis=1)
print(punct)
print(capit)

# For English PnC model punctuation output for each token is: { none=0, comma=1, dot=2, question=3 }
# Capitlization is: { uppercase=1, lowercase=0 }
# This info can be found when unzipping .nemo model file, and locating .csv files (e.g. fe160f3a917d411b99852e509e3279a3_punct_label_ids.csv)

out = ""
for i in range(len(tokens)):
    t = tokens[i]
    out += tokenizer.decode(t)
    #out += str(punct[i])
    out += str(capit[i])

print(out)

session ok
22 [101, 13973, 2003, 1037, 2530, 7472, 2653, 1998, 2003, 1996, 6284, 1998, 2880, 2653, 1997, 2093, 8392, 4279, 1999, 2789, 3577, 102]
Average inference time: 0.031 sec
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 0]
[0 1 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0]
[CLS]0catalan1is0a0western1romance1language0and0is0the0indigenous0and0official0language0of0three0autonomous0communities0in0eastern0spain1[SEP]0
